<a href="https://colab.research.google.com/github/Manawariqbal/LLM-FIneTuning/blob/main/Efficient_Fine_Tuning_with_PEFT_LoRA_(Recommended_for_Real_LLMs).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# !pip install transformers peft datasets accelerate bitsandbytes  # bitsandbytes for optional QLoRA

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [4]:
model_name = "HuggingFaceTB/SmolLM2-135M"  # Small modern causal LM (or try gpt2, Qwen2.5-0.5B, etc.)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  269MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [7]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 32.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [8]:
# LoRA config — only train ~0.1–1% of parameters
lora_config = LoraConfig(
    r=8,                        # Rank (higher = more capacity, more params)
    lora_alpha=16,              # Scaling factor
    target_modules="all-linear",  # Apply to all linear layers (or specify ["q_proj", "v_proj"])
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # You should see something like 0.1–1% trainable

trainable params: 2,442,240 || all params: 136,957,248 || trainable%: 1.7832


In [10]:
# Prepare a small instruction-style or domain dataset
# Example: a tiny custom dataset or subset of an instruction dataset
dataset = load_dataset("databricks/databricks-dolly-15k", split="train[:200]")  # tiny demo

def format_example(example):
    text = f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['response']}"
    return {"text": text}

dataset = dataset.map(format_example)
def tokenize(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256, padding="max_length")

tokenized = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [12]:
from transformers import DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./lora-finetuned",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,          # Higher LR is common with LoRA
    logging_steps=10,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer.train()

Step,Training Loss
10,2.673953
20,2.454392


TrainOutput(global_step=25, training_loss=2.5514771270751955, metrics={'train_runtime': 28.6461, 'train_samples_per_second': 6.982, 'train_steps_per_second': 0.873, 'total_flos': 33375957811200.0, 'train_loss': 2.5514771270751955, 'epoch': 1.0})

In [13]:
# Save only the adapter (very small)
model.save_pretrained("./lora-adapter")

# Later: load base + adapter for inference
# from peft import PeftModel
# base = AutoModelForCausalLM.from_pretrained(model_name)
# model = PeftModel.from_pretrained(base, "./lora-adapter")